In [2]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [5]:
# llm 
model = ChatGroq(
    model ="llama-3.3-70b-versatile",
    temperature=0
)

In [6]:
# define state
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [7]:
# define task
def chat_node(state : ChatState):
    # user query
    messages = state['messages']

    # query --> llm
    response = model.invoke(messages)

    return {'messages': [response]}

In [8]:
# build graph
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile()

In [9]:
initial_state = {
    'messages': [HumanMessage(content='What is the national animal of Bangladesh?')]
}

chatbot.invoke(initial_state)['messages'][-1].content

'The national animal of Bangladesh is the Royal Bengal Tiger.'